# NOVI PODATOCI

In [1]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import random
import glob

In [2]:
root = r"C:\Users\kizo\Desktop\emteq stari\s11_processed"
subjects = [d for d in os.listdir(root) if os.path.isdir(os.path.join(root, d))]

In [3]:
subjects

['AleksandraK',
 'AnaI',
 'AndrejCh',
 'AndrejJ',
 'AngelaK',
 'AngelaN',
 'BojanD',
 'DavidN',
 'ElenaA',
 'HristijanD',
 'IrinaP',
 'IvaI',
 'IvanP',
 'KristinaK',
 'LeoJ',
 'MarijaB',
 'MartinD',
 'MatrinI',
 'MihaelaD',
 'MilaP',
 'MonikaS',
 'PavelT',
 'RezeartaLj',
 'RistoK',
 'Sandra',
 'SandraM',
 'SandraSh',
 'SaraI',
 'SaraSh',
 'SlavicaSh',
 'StefanI',
 'StefanijaK',
 'StefanIl',
 'StojanV',
 'Teodora',
 'TeodoraD',
 'TeodorS',
 'TomiN',
 'VladimirK']

In [4]:
subject_csv = {}

for s in subjects:
    subj_dir = os.path.join(root, s)
    csvs = glob.glob(os.path.join(subj_dir, "*_eating_processed.csv"))

    if len(csvs) == 0:
        print(f" NO eating_processed for {s}")
    elif len(csvs) > 1:
        print(f" MORE than one for {s}, using first")
        subject_csv[s] = csvs[0]
    else:
        subject_csv[s] = csvs[0]

print("Total subjects with eating_processed:", len(subject_csv))

Total subjects with eating_processed: 39


In [5]:
random.seed(42)
train_subjects = subjects[:27]
val_subjects   = subjects[27:33]
test_subjects  = subjects[33:39]

print("TRAIN:", len(train_subjects), train_subjects)
print("VAL:",   len(val_subjects), val_subjects)
print("TEST:",  len(test_subjects), test_subjects))

TRAIN: 27 ['RistoK', 'StojanV', 'AndrejJ', 'IvaI', 'IrinaP', 'TeodoraD', 'SaraSh', 'StefanI', 'StefanijaK', 'RezeartaLj', 'SandraSh', 'PavelT', 'IvanP', 'AngelaK', 'HristijanD', 'Teodora', 'Sandra', 'SlavicaSh', 'MonikaS', 'SandraM', 'TeodorS', 'SaraI', 'MilaP', 'MartinD', 'VladimirK', 'StefanIl', 'AndrejCh']
VAL: 6 ['AleksandraK', 'TomiN', 'KristinaK', 'MihaelaD', 'AngelaN', 'BojanD']
TEST: 6 ['ElenaA', 'LeoJ', 'MarijaB', 'MatrinI', 'AnaI', 'DavidN']


In [6]:
subject_csv = {}

for s in subjects:
    subj_dir = os.path.join(root, s)
    csvs = glob.glob(os.path.join(subj_dir, "*_eating_processed.csv"))

    if len(csvs) == 0:
        print(f" NO eating_processed for {s}")
    elif len(csvs) > 1:
        print(f" MORE than one for {s}, using first")
        subject_csv[s] = csvs[0]
    else:
        subject_csv[s] = csvs[0]

print("Total subjects with eating_processed:", len(subject_csv))

Total subjects with eating_processed: 39


In [7]:
def load_subject_df(subject):
    csv_path = subject_csv[subject]
    df = pd.read_csv(csv_path)
    df["subject"] = subject
    return df

In [8]:
train_dfs = [load_subject_df(s) for s in train_subjects if s in subject_csv]
val_dfs   = [load_subject_df(s) for s in val_subjects   if s in subject_csv]
test_dfs  = [load_subject_df(s) for s in test_subjects  if s in subject_csv]

print(len(train_dfs), len(val_dfs), len(test_dfs))


27 6 6


In [9]:
train_df = pd.concat(train_dfs, ignore_index=True)
val_df   = pd.concat(val_dfs,   ignore_index=True)
test_df  = pd.concat(test_dfs,  ignore_index=True)

print(train_df.shape, val_df.shape, test_df.shape)
print(train_df["subject"].nunique(),
      val_df["subject"].nunique(),
      test_df["subject"].nunique())

(893451, 24) (186001, 24) (174005, 24)
27 6 6


In [10]:
cols_to_drop = [
    "Accelerometer Z",
    "Gyroscope Z",
    "Rotation Vector Z",
    "Rotation Vector W",
    "Magnetometer X",
    "Magnetometer Y",
    "Magnetometer Z",
    "Pressure",
    "Software Timestamp",   # 👈 DROP
    "Hardware Timestamp",   # 👈 DROP (ако постои)
]

train_df = train_df.drop(columns=cols_to_drop, errors="ignore")
val_df   = val_df.drop(columns=cols_to_drop, errors="ignore")
test_df  = test_df.drop(columns=cols_to_drop, errors="ignore")


In [11]:
print(train_df.shape, val_df.shape, test_df.shape)
print(train_df["subject"].nunique(),
      val_df["subject"].nunique(),
      test_df["subject"].nunique())

(893451, 14) (186001, 14) (174005, 14)
27 6 6


In [12]:

def make_windows_from_df(df, feature_cols, label_col="Annotations", subject_col="subject",
                         time_col="Software Timestamp",
                         window_size=150, step_size=25, label_mode="last"):
    X_list, y_list = [], []

    for sid, g in df.groupby(subject_col):
        if time_col in g.columns:
            g = g.sort_values(time_col)

        X = g[feature_cols].to_numpy(dtype="float32")
        y = g[label_col].to_numpy(dtype="int32")

        n = len(g)
        for start in range(0, n - window_size + 1, step_size):
            end = start + window_size
            xw = X[start:end]

            if label_mode == "last":
                yw = y[end - 1]
            else:  # majority
                vals, counts = np.unique(y[start:end], return_counts=True)
                yw = vals[np.argmax(counts)]

            X_list.append(xw)
            y_list.append(yw)

    return np.stack(X_list), np.array(y_list, dtype="int32")

In [ ]:
label_col   = "Annotations"
subject_col = "subject"
time_col    = "Software Timestamp" 

exclude = {label_col, subject_col, "Software Timestamp", "Hardware Timestamp"}
feature_cols = [c for c in train_df.columns if c not in exclude]

print("n_features =", len(feature_cols))
print("first features:", feature_cols[:10])


n_features = 12
first features: ['Accelerometer X', 'Accelerometer Y', 'Gyroscope X', 'Gyroscope Y', 'Rotation Vector X', 'Rotation Vector Y', 'Navigation Right Temple X', 'Navigation Right Temple Y', 'Proximity Right Temple', 'Navigation Right Temple IQ']


In [14]:
window_sec = 4.0
step_sec = 0.5 
fs = 50 #50hz

window_size = int(fs * window_sec)
step_size   = int(fs * step_sec)

X_train, y_train = make_windows_from_df(train_df, feature_cols, label_col, subject_col, time_col,
                                        window_size, step_size, label_mode="last")
X_val, y_val     = make_windows_from_df(val_df, feature_cols, label_col, subject_col, time_col,
                                        window_size, step_size, label_mode="last")
X_test, y_test   = make_windows_from_df(test_df, feature_cols, label_col, subject_col, time_col,
                                        window_size, step_size, label_mode="last")

print("Train:", X_train.shape, y_train.shape)
print("Val:  ", X_val.shape, y_val.shape)
print("Test: ", X_test.shape, y_test.shape)


Train: (35540, 200, 12) (35540,)
Val:   (7396, 200, 12) (7396,)
Test:  (6917, 200, 12) (6917,)


In [15]:
t = pd.to_datetime(train_dfs[0]["Hardware Timestamp"], errors="coerce")
dt = t.diff().dt.total_seconds().dropna()
dt = dt[(dt > 0) & (dt < 1)]
FS = 1.0 / dt.median()
print("FS ≈", FS)

FS ≈ 49999.99999999999


# Z SKALIRANJE

In [16]:
mean = X_train.mean(axis=(0,1), keepdims=True)
std  = X_train.std(axis=(0,1), keepdims=True)
std[std == 0] = 1

X_train_z = (X_train - mean) / std
X_val_z   = (X_val   - mean) / std
X_test_z  = (X_test  - mean) / std

print("train mean first 5 feats:", X_train_z.mean(axis=(0,1))[:5])
print("train std  first 5 feats:", X_train_z.std(axis=(0,1))[:5])


train mean first 5 feats: [-5.1918738e-07  1.8609819e-05 -1.7769363e-09 -3.5957333e-09
 -3.2127823e-06]
train std  first 5 feats: [1.0000019  0.9999974  0.99999774 1.0000024  1.0000029 ]


In [ ]:
classes, counts = np.unique(y_train, return_counts=True)
print("y_train counts:", dict(zip(classes, counts)))
print("Eating %:", y_train.mean() * 100)

n0 = counts[classes.tolist().index(0)]
n1 = counts[classes.tolist().index(1)]
class_weight = {0: 1.0, 1: (n0 / n1)}
print("class_weight:", class_weight)


y_train counts: {np.int32(0): np.int64(14429), np.int32(1): np.int64(21111)}
Eating %: 59.40067529544175
class_weight: {0: 1.0, 1: np.float64(0.6834825446449718)}


In [18]:
import tensorflow as tf
from tensorflow.keras import layers, models

In [19]:
input_shape = (X_train_z.shape[1], X_train_z.shape[2])  # (200, 12)

model = models.Sequential([
    layers.Input(shape=input_shape),

    layers.Conv1D(64, 7, padding="same"),
    layers.BatchNormalization(),
    layers.ReLU(),
    layers.MaxPooling1D(2),
    layers.Dropout(0.2),

    layers.Conv1D(128, 5, padding="same"),
    layers.BatchNormalization(),
    layers.ReLU(),
    layers.MaxPooling1D(2),
    layers.Dropout(0.2),

    layers.Conv1D(256, 3, padding="same"),
    layers.BatchNormalization(),
    layers.ReLU(),
    layers.GlobalAveragePooling1D(),
    layers.Dropout(0.3),

    layers.Dense(64, activation="relu"),
    layers.Dropout(0.3),
    layers.Dense(1, activation="sigmoid")
])

model.compile(
    optimizer=tf.keras.optimizers.Adam(1e-3),
    loss="binary_crossentropy",
    metrics=[
        "accuracy",
        tf.keras.metrics.Precision(name="precision"),
        tf.keras.metrics.Recall(name="recall"),
        tf.keras.metrics.AUC(name="auc"),
    ]
)

model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                         ┃ Output Shape                ┃         Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━┩
│ conv1d (Conv1D)                      │ (None, 200, 64)             │           5,440 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ batch_normalization                  │ (None, 200, 64)             │             256 │
│ (BatchNormalization)                 │                             │                 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ re_lu (ReLU)                         │ (None, 200, 64)             │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ max_pooling1d (MaxPooling1D)         │ (None, 100, 64)             │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dropout (Dropout)                    │ (None, 100, 64)             │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ conv1d_1 (Conv1D)                    │ (None, 100, 128)            │          41,088 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ batch_normalization_1                │ (None, 100, 128)            │             512 │
│ (BatchNormalization)                 │                             │                 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ re_lu_1 (ReLU)                       │ (None, 100, 128)            │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ max_pooling1d_1 (MaxPooling1D)       │ (None, 50, 128)             │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dropout_1 (Dropout)                  │ (None, 50, 128)             │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ conv1d_2 (Conv1D)                    │ (None, 50, 256)             │          98,560 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ batch_normalization_2                │ (None, 50, 256)             │           1,024 │
│ (BatchNormalization)                 │                             │                 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ re_lu_2 (ReLU)                       │ (None, 50, 256)             │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ global_average_pooling1d             │ (None, 256)                 │               0 │
│ (GlobalAveragePooling1D)             │                             │                 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dropout_2 (Dropout)                  │ (None, 256)                 │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense (Dense)                        │ (None, 64)                  │          16,448 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dropout_3 (Dropout)                  │ (None, 64)                  │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_1 (Dense)                      │ (None, 1)                   │              65 │
└──────────────────────────────────────┴─────────────────────────────┴─────────────────┘

 Total params: 163,393 (638.25 KB)

 Trainable params: 162,497 (634.75 KB)

 Non-trainable params: 896 (3.50 KB)

In [ ]:
cb = [
    tf.keras.callbacks.EarlyStopping(monitor="val_auc", mode="max",
                                     patience=8, restore_best_weights=True),
    tf.keras.callbacks.ReduceLROnPlateau(monitor="val_auc", mode="max",
                                         factor=0.5, patience=3, min_lr=1e-6)
]

history = model.fit(
    X_train_z, y_train,
    validation_data=(X_val_z, y_val),
    epochs=60,
    batch_size=256,
    # class_weight=class_weight,   
    callbacks=cb,
    verbose=1
)

Epoch 1/60
139/139 ━━━━━━━━━━━━━━━━━━━━ 17s 108ms/step - accuracy: 0.8484 - auc: 0.9190 - loss: 0.3541 - precision: 0.8568 - recall: 0.8943 - val_accuracy: 0.7477 - val_auc: 0.8992 - val_loss: 0.5151 - val_precision: 0.9056 - val_recall: 0.6238 - learning_rate: 0.0010
Epoch 2/60
139/139 ━━━━━━━━━━━━━━━━━━━━ 15s 110ms/step - accuracy: 0.8901 - auc: 0.9530 - loss: 0.2687 - precision: 0.8947 - recall: 0.9236 - val_accuracy: 0.8225 - val_auc: 0.9061 - val_loss: 0.4173 - val_precision: 0.8076 - val_recall: 0.9052 - learning_rate: 0.0010
Epoch 3/60
139/139 ━━━━━━━━━━━━━━━━━━━━ 16s 112ms/step - accuracy: 0.8995 - auc: 0.9591 - loss: 0.2510 - precision: 0.9025 - recall: 0.9315 - val_accuracy: 0.8344 - val_auc: 0.9084 - val_loss: 0.4143 - val_precision: 0.8404 - val_recall: 0.8768 - learning_rate: 0.0010
Epoch 4/60
139/139 ━━━━━━━━━━━━━━━━━━━━ 17s 120ms/step - accuracy: 0.9072 - auc: 0.9638 - loss: 0.2357 - precision: 0.9090 - recall: 0.9377 - val_accuracy: 0.8579 - val_auc: 0.9257 - val_loss: 

In [ ]:
from sklearn.metrics import confusion_matrix, classification_report

# default threshold 0.5 
y_prob = model.predict(X_test_z).ravel()
y_pred = (y_prob >= 0.64).astype(int)

cm = confusion_matrix(y_test, y_pred)
print("Confusion matrix:\n", cm)
print(classification_report(y_test, y_pred, digits=4))


217/217 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step 
Confusion matrix:
 [[2674  508]
 [ 407 3328]]
              precision    recall  f1-score   support

           0     0.8679    0.8404    0.8539      3182
           1     0.8676    0.8910    0.8791      3735

    accuracy                         0.8677      6917
   macro avg     0.8677    0.8657    0.8665      6917
weighted avg     0.8677    0.8677    0.8675      6917



In [25]:
from sklearn.metrics import roc_curve

fpr, tpr, thr = roc_curve(y_test, y_prob)
best = thr[(tpr - fpr).argmax()]
print(best)


0.6397457
